## 第17章 线程和并行

### 1.线程基础

- **线程(Thread)**：程序中单一的指令执行序列，完成I/O阻塞型任务有用，完成CPU阻塞型任务无用。**同一进程内同一时刻只能运行一个线程(GIL)**。
    - 主要类型：
        - 单线程程序：没有并发/多进程的Python程序都在单线程中运行。
        - 多线程程序：在同一进程中同时运行多个线程以实现并发。
    - 与异步对比：
        - 线程：抢占式多任务，由操作系统调度，操作系统强制抢占当前线程，数据共享难。
        - 异步：协作式多任务，由Python自身调度，程序自愿让出控制权，数据共享相对容易。
    - 实现方法：
        - threading模块
            - `threading.Thread(target=func, args=(), daemon=True)`：创建一个线程对象。
                - `func`：要线程执行的函数，**注意：该函数不能直接返回值给调用者**。
                - `args`：函数的参数。必须是元组，单个参数时需要在元组中添加逗号。
                - `daemon=True`：表示该线程是一个守护线程，当主线程退出时，守护线程也会被强制退出。
            - `thread.start()`：启动线程。
            - `thread.is_alive()`：判断线程是否还在运行。
            - `thread.join(timeout=None)`：阻塞等待线程结束。
                - `timeout`：指定等待时间，超过后不管线程是否结束都继续执行。
        - concurrent.futures模块
            - `concurrent.futures.ThreadPoolExecutor()`：创建线程池。可以通过`with`管理线程池的生命周期。
            - `executor.submit(fn, *args)`：在线程池中创建一个线程。返回`Future`对象。
            - `executor.map(fn, iterable)`：批量创建线程，返回结果迭代器。
            - `future.result()`：阻塞获取结果。
            - `future.done()`：判断线程是否执行完成。
            - `future.cancel()`：取消线程执行。
            - `executor.shutdown()`：关闭线程池。
    - 向线程传递数据：
        - 状态共享(全局变量)：最简单的方法，也是最危险的方法，容易触发条件竞争，导致数据不一致。
        - future对象：通过`executor.submit()`返回的future对象，可以获取线程执行的结果。
        - 消息队列：使用`queue.Queue()`创建一个消息队列，线程之间通过队列传递消息，是最安全的方法，但系统开销较大。

In [ ]:
# 通用部分代码(必须先运行)
BOUND = 10**5

def collatz(n):
    steps = 0
    while n > 1:
        if n % 2:
            n = 3 * n + 1
        else:
            n //= 2
        steps += 1
    return steps

def length_counter(target):
    count = 0
    for n in range(2, BOUND):
        if collatz(n) == target:
            count += 1
    return count

In [ ]:
# 通过threading模块创建线程
import threading

guess = None  # 通过全局变量向线程传递数据

def get_input_global(prompt):
    global guess  # 读取或设置全局变量
    while True:
        n = input(prompt)
        try:
            n = int(n)
        except ValueError:
            print("请输入一个整数")
            continue
        if n <= 0:
            print("请输入一个大于0的整数")
        else:
            guess = n
            return n

def main_thread():
    target = get_input_global("请输入Collatz序列的计算步数：")

    t_guess = threading.Thread(                         # 创建线程
        target=get_input_global,                               # 线程执行的函数
        args=("请输入你猜的Collatz序列的个数：",),          # 线程执行的函数的参数，必须为元组(加逗号)
        daemon=True                                     # 指定线程为守护线程，主线程结束时自动结束
    )
    t_guess.start()                                     # 启动线程
    count = length_counter(target)
    t_guess.join(timeout=3)                             # 等待线程执行完成，指定等待时间，超过时间后继续下步执行
    if t_guess.is_alive():                              # 检查线程是否还在执行
        print("线程未执行完成，超时")
        return

    if guess == count:
        print("恭喜你猜对了！")
    else:
        print(f"很遗憾，你猜错了。实际的数是{count}")

if __name__ == "__main__":
    main_thread()

In [ ]:
# 通过concurrent.futures模块创建线程
import concurrent.futures

def get_input_normal(prompt):
    while True:
        value = input(prompt)
        try:
            value = int(value)
        except ValueError:
            print("请输入一个整数")
            continue
        if value <= 0:
            print("请输入一个大于0的整数")
        else:
            return value

def main_futures():
    target = get_input_normal("请输入Collatz序列的计算步数：")

    executor = concurrent.futures.ThreadPoolExecutor()      # 创建线程池
    future = executor.submit(                               # 在线程池中创建一个线程
        get_input_normal,                                          # 线程要执行的函数
        "请输入你猜的Collatz序列的个数：",                      # 线程要执行的函数的参数
    )
    count = length_counter(target)
    guess = future.result()                                 # 等待线程执行完成，获取线程执行的结果
    executor.shutdown()                                     # 关闭线程池

    if guess == count:
        print("恭喜你猜对了！")
    else:
        print(f"很遗憾，你猜错了。实际的数是{count}")

if __name__ == "__main__":
    main_futures()

In [ ]:
import concurrent.futures

def main_with_futures():
    target = get_input_normal("请输入Collatz序列的计算步数：")

    # 通过with语句创建线程池，确保线程池在使用后被正确关闭
    with concurrent.futures.ThreadPoolExecutor() as executor:
        future = executor.submit(get_input_normal, "请输入你猜的Collatz序列的个数：")
        count = length_counter(target)
        guess = future.result()

    if guess == count:
        print("恭喜你猜对了！")
    else:
        print(f"很遗憾，你猜错了。实际的数是{count}")

if __name__ == "__main__":
    main_with_futures()

### 2.锁

- **竞态条件**：多个线程同时访问/修改共享数据，导致结果取决于线程的执行顺序。其本质是操作不是**原子性**的——看似一步完成的操作实际由多步组成。
- **锁**：锁是最基本的同步原语：**同一时刻只允许一个线程进入临界区**。
    - `thread.Lock()`：创建一个锁对象。
    - `lock.acquire()`：获取锁，如果锁已被其他线程占用，则阻塞等待。
    - `lock.release()`：释放锁，允许其他线程获取锁。
    - `with lock: ...`：自动获取锁并确保在退出时释放锁。推荐使用。

In [ ]:
# 错误版本(因竞争条件导致结果不准确)
import time
import concurrent.futures
import functools

BOUND = 10**5

# 计数器
class CounterErr:
    count = 0  # 通过类属性存储不同线程之间的共享状态

    @classmethod
    def increment(cls):
        value = cls.count + 1
        time.sleep(0.1)         # 故意产生竞争条件
        cls.count = value

    @classmethod
    def get(cls):
        return cls.count

    @classmethod
    def reset(cls):
        cls.count = 0

def collatz_err(target, n):
    steps = 0
    while n > 1:
        if n % 2:
            n = n * 3 + 1
        else:
            n //= 2
        steps += 1
    if steps == target:
        CounterErr.increment()  # 通过类来共享计算结果

def length_counter_err(target):
    CounterErr.reset()                                                      # 重置计数器
    executor = concurrent.futures.ThreadPoolExecutor(max_workers=5)         # 创建线程池，最大线程数为5
    func = functools.partial(collatz_err, target)                           # 创建偏函数，固定target参数
    executor.map(func, range(2, BOUND))                                     # 迭代提交任务到线程池，map方法只能提交单个参数
    executor.shutdown()                                                     # 关闭线程池
    return CounterErr.get()                                                 # 获取计数器值(因多个线程同时修改，导致结果不准确)

# 获取用户输入
def get_input_err(prompt):
    while True:
        value = input(prompt)
        try:
            value = int(value)
        except ValueError:
            print("请输入一个整数")
            continue
        if value <= 0:
            print("请输入一个大于0的整数")
        else:
            return value

def main_err():
    target = get_input_err("请输入Collatz序列的计算步数：")

    with concurrent.futures.ThreadPoolExecutor() as executor:
        future = executor.submit(get_input_err, "请输入你猜的Collatz序列的个数：")
        count = length_counter_err(target)
        guess = future.result()

    if guess == count:
        print("恭喜你猜对了！")
    else:
        print(f"很遗憾，你猜错了。实际的数是{count}")

if __name__ == "__main__":
    main_err()

In [ ]:
# 使用锁避免竞态条件
import concurrent.futures
import functools
import threading

BOUND = 10**5

# 计数器
class CounterLock:
    count = 0                       # 通过类属性存储不同线程之间的共享状态
    _lock = threading.Lock()        # 创建锁对象

    @classmethod
    def increment(cls):
        with cls._lock:             # 加锁
            value = cls.count + 1
            cls.count = value

    @classmethod
    def get(cls):
        return cls.count

    @classmethod
    def reset(cls):
        cls.count = 0

def collatz_lock(target, n):
    steps = 0
    while n > 1:
        if n % 2:
            n = n * 3 + 1
        else:
            n //= 2
        steps += 1
    if steps == target:
        CounterLock.increment()

def length_counter_lock(target):
    CounterLock.reset()
    executor = concurrent.futures.ThreadPoolExecutor(max_workers=5)
    func = functools.partial(collatz_lock, target)
    executor.map(func, range(2, BOUND))
    executor.shutdown()
    return CounterLock.get()

def get_input_lock(prompt):
    while True:
        value = input(prompt)
        try:
            value = int(value)
        except ValueError:
            print("请输入一个整数")
            continue
        if value <= 0:
            print("请输入一个大于0的整数")
        else:
            return value

def main_lock():
    target = get_input_lock("请输入Collatz序列的计算步数：")

    with concurrent.futures.ThreadPoolExecutor() as executor:
        future = executor.submit(get_input_lock, "请输入你猜的Collatz序列的个数：")
        count = length_counter_lock(target)
        guess = future.result()

    if guess == count:
        print("恭喜你猜对了！")
    else:
        print(f"很遗憾，你猜错了。实际的数是{count}")

if __name__ == "__main__":
    main_lock()

- **死锁**：两个或多个线程互相等待对方释放锁，导致全部永久阻塞。
    - 经典场景：
        - 线程1：持有锁A，等待锁B。
        - 线程2：持有锁B，等待锁A。
        - → 两个线程永远等下去。
    - 死锁条件：
        - 互斥：资源一次只能被一个线程使用。
        - 持有并等待：线程持有至少一个资源，同时等待其他资源。
        - 不可抢占：资源不能被强制夺走。
        - 循环等待：等待链形成闭环。
    - 预防策略：
        - 超时机制：使用`lock.acquire(timeout=X)`，超时则放弃。
        - 统一锁顺序：所有线程按相同顺序获取多把锁。
        - 尽量减少锁的数量：能用一把锁就不用两把锁。


### 3.用队列传递消息

通常使用队列(`queue`模块)在线程之间传递消息，一个或多个线程可以将数据推送到队列，而一个或多个其他线程可以从队列中获取数据。

In [ ]:
# 通过队列传递结果
import concurrent.futures
import functools
import queue

BOUND = 10**5

def collatz_queue(results, n):
    steps = 0
    while n > 1:
        if n % 2:
            n = n * 3 + 1
        else:
            n //= 2
        steps += 1
    results.put(steps)  # 将结果放入队列

def length_counter_queue(target):
    results = queue.Queue()  # 创建一个队列用于存储结果
    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
        func = functools.partial(collatz_queue, results)
        executor.map(func, range(2, BOUND))
    results = list(results.queue)  # 将队列中的结果转换为列表
    return results.count(target)  # 统计等于目标值的个数

def get_input_queue(prompt):
    while True:
        value = input(prompt)
        try:
            value = int(value)
        except ValueError:
            print("请输入一个整数")
            continue
        if value <= 0:
            print("请输入一个大于0的整数")
        else:
            return value

def main_queue():
    target = get_input_queue("请输入Collatz序列的计算步数：")

    with concurrent.futures.ThreadPoolExecutor() as executor:
        future = executor.submit(get_input_queue, "请输入你猜的Collatz序列的个数：")
        count = length_counter_queue(target)
        guess = future.result()

    if guess == count:
        print("恭喜你猜对了！")
    else:
        print(f"很遗憾，你猜错了。实际的数是{count}")

if __name__ == "__main__":
    main_queue()

### 4.多线程future

In [ ]:
# 这是最简捷、开销最小的方案
import concurrent.futures

BOUND = 10**5

def collatz_futures(n):
    steps = 0
    while n > 1:
        if n % 2:
            n = n * 3 + 1
        else:
            n //= 2
        steps += 1
    return steps

def length_counter_futures(target):
    count = 0
    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
        for result in executor.map(collatz_futures, range(2, BOUND)):  # future避免锁问题
            if result == target:
                count += 1
    return count

def get_input_futures(prompt):
    while True:
        value = input(prompt)
        try:
            value = int(value)
        except ValueError:
            print("请输入一个整数")
            continue
        if value <= 0:
            print("请输入一个大于0的整数")
        else:
            return value

def main_futures():
    target = get_input_futures("请输入Collatz序列的计算步数：")

    with concurrent.futures.ThreadPoolExecutor() as executor:
        future = executor.submit(get_input_futures, "请输入你猜的Collatz序列的个数：")
        count = length_counter_futures(target)
        guess = future.result()

    if guess == count:
        print("恭喜你猜对了！")
    else:
        print(f"很遗憾，你猜错了。实际的数是{count}")

if __name__ == "__main__":
    main_futures()

### 5.多进程

- **线程和进程对比：**

|      | 线程(Thread)           | 进程(Process)                    |
| ---- | ------------------------ | ---------------------------------- |
| 内存 | 共享同一进程的内存空间   | **各自独立**的内存空间             |
| GIL  | 受GIL限制                | 每个进程有独立GIL → **可实现并行** |
| 通信 | 直接共享变量（需锁保护） | 必须通过IPC（队列/管道等）         |
| 开销 | 轻量                     | 重量级（启动慢、占内存多）         |
| 适用 | IO密集型任务                 | CPU密集型任务                 |


- **创建进程：**
    - `multiprocessing`模块:
        - `multiprocessing.Process()`：创建进程对象。
        - `process.start()`：启动进程。
        - `process.join()`：等待进程执行完成。
        - `process.terminate()`：强制终止进程。
    - `concurrent.futures`模块:
        - `ProcessPoolExecutor()`：创建进程池器对象。
        - `executor.submit(fn, *args)`：提交任务到进程池器。
        - `executor.map(fn, iterable, chunksize=None)`：批量提交任务到进程池器，chunksize指定每个进程处理的任务数量。
        - `executor.shutdown()`：关闭进程池器。
        - `executor.result()`：获取任务执行结果。
    
示例：[mod/multi.py](mod/multi.py)


In [ ]:
# 因为multi.py中使用了进程，在Jupyter Notebook中运行会报错，只能以独立的Python进程运行
%run mod/multi.py

### 6.生产者和消费者问题

代码示例：[mod/producer.py](mod/producer.py)

In [ ]:
%run mod/producer.py

### 7.本章小结

- **核心知识脉络**

```text
线程与并行
│
├── 1. 核心概念区分 ⭐⭐
│   ├── 并发（Concurrency）= 多任务交替，提升响应性
│   ├── 并行（Parallelism）= 多任务同时，提升速度
│   ├── 线程 = 抢占式多任务（OS管理切换）
│   └── 异步 = 协作式多任务（代码主动让出）
│
├── 2. 线程基础（Threading）
│   ├── threading.Thread(target, args, daemon)
│   ├── start() / join() / is_alive()
│   ├── 守护线程（daemon=True）→ 主程序退出时强制终止
│   └── 线程函数不能直接返回值 → 需其他通信机制
│
├── 3. 共享数据的危险 ⭐⭐
│   ├── 竞态条件（Race Condition）
│   │   ├── 多线程同时访问/修改共享数据
│   │   ├── 结果取决于线程执行顺序（不确定性）
│   │   └── Heisenbug：观察即改变
│   └── 银行账户类比
│
├── 4. 锁（Locks）⭐⭐
│   ├── Lock = 同一时刻只允许一个线程进入临界区
│   ├── with lock: → 推荐用法（自动获取/释放）
│   ├── RLock = 可重入锁（同一线程可多次获取）
│   └── ⚠️ 锁的问题：遗忘/粒度/死锁
│
├── 5. 死锁（Deadlock）⭐⭐
│   ├── 两个+线程互相等待对方释放锁
│   ├── 四个必要条件：互斥/持有并等待/不可抢占/循环等待
│   └── 预防：统一锁顺序/超时/减少锁数量
│
├── 6. Queue —— 线程安全队列 ⭐⭐
│   ├── Queue（FIFO）/ LifoQueue（LIFO）/ PriorityQueue
│   ├── put() / get() / task_done() / join()
│   └── 内部已加锁，无需手动管理
│
├── 7. Event —— 线程间信号 ⭐⭐
│   ├── set() / clear() / wait() / is_set()
│   └── 比轮询高效（零开销等待）
│
├── 8. concurrent.futures ⭐⭐
│   ├── ThreadPoolExecutor（线程池）
│   ├── ProcessPoolExecutor（进程池）
│   ├── submit() → Future
│   ├── map() → 批量提交
│   └── as_completed() → 按完成顺序获取
│
├── 9. 多进程（Multiprocessing）⭐⭐
│   ├── 每个进程有独立GIL → 真正并行
│   ├── 独立内存空间 → 必须IPC通信
│   ├── multiprocessing.Queue / Event
│   └── 开销大（启动慢、占内存）
│
├── 10. 生产者-消费者模式 ⭐
│   ├── Producer → Queue → Consumer
│   ├── exit_event通知消费者退出
│   └── timeout防止消费者永久阻塞
│
└── 11. 实用忠告
    ├── 三种多任务方式：异步/线程/多进程
    ├── 异步+线程 → IO-bound；多进程 → CPU-bound
    ├── 并发/并行不是万能药 → 增加复杂度和bug风险
    └── 不需要就不要用 → 同步代码没问题
```

- **警告与提示汇总表**

| 类型   | 内容                                                         |
| ------ | ------------------------------------------------------------ |
| ⚠️ 警告 | 线程函数**不能直接返回值**给调用者，需其他通信机制           |
| ⚠️ 警告 | 全局变量+多线程 = **竞态条件**风险                           |
| ⚠️ 警告 | 竞态条件的结果**不确定**——每次运行可能不同                   |
| ⚠️ 警告 | 手动`acquire()`/`release()`锁，若临界区抛异常可能**忘记释放** → 死锁 |
| ⚠️ 警告 | 多把锁的获取顺序不一致 → **死锁**                            |
| ⚠️ 警告 | `queue.Queue()`只用于线程间！多进程必须用`multiprocessing.Queue()` |
| ⚠️ 警告 | 守护线程被强制终止时**不会优雅清理**                         |
| ⚠️ 警告 | 多进程的启动和数据序列化有**显著开销**，数据量小时可能比单进程更慢 |
| ⚠️ 警告 | 并发/并行不是万能药——增加复杂度和bug风险                     |
| ⚠️ 警告 | 书中生产者-消费者代码的设计**不一定适用于你的场景**          |
| 💡 技巧 | 使用`with lock:`而非手动`acquire()`/`release()`              |
| 💡 技巧 | Event比轮询高效（零开销等待）                                |
| 💡 技巧 | 异步库中有与线程类似的锁、事件、池、Future                   |
| 💡 技巧 | 能用线程实现的，大多数也能用异步实现                         |
| 💡 技巧 | 不需要额外能力时，避免额外复杂度——同步代码没问题             |
| 💡 技巧 | 多进程调试用带时间戳和进程ID的日志系统                       |
| 💡 技巧 | 预防死锁：统一锁获取顺序 / 使用超时 / 减少锁数量             |

- **三种多任务方式对比**

|            | 异步         | 线程               | 多进程                  |
| ---------- | -------------- | ---------------- | ----------------------- |
| 任务类型   | IO密集型       | IO密集型         | CPU密集型               |
| 并发/并行  | 并发           | 并发             | 并行                    |
| 多任务方式 | 协作式         | 抢占式           | 独立进程                |
| 管理者     | Python自身     | 操作系统         | 操作系统                |
| GIL        | 受限           | 受限             | 不受限（每进程独立GIL） |
| 内存       | 共享           | 共享（需锁保护） | 独立（需IPC）           |
| 开销       | 最低           | 低               | 高                      |
| 复杂度     | 中             | 高（竞态/死锁）  | 高（IPC/序列化）        |
